In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')

In [2]:
def generate_synthetic_dataset(n_normal: int = 10000, n_attack: int = 3000) -> pd.DataFrame:
    """
    Generate synthetic data with the SAME feature structure as UNSW-NB15.
    This lets you test the full pipeline before getting the real dataset.

    Real UNSW-NB15 features we replicate:
      - dur:        Flow duration
      - sbytes:     Source to destination bytes
      - dbytes:     Destination to source bytes
      - sttl:       Source to destination TTL
      - dttl:       Destination to source TTL
      - sloss:      Source packets retransmitted/dropped
      - dloss:      Destination packets retransmitted/dropped
      - sload:      Source bits per second
      - dload:      Destination bits per second
      - spkts:      Source to destination packet count
      - dpkts:      Destination to source packet count
      - sinpkt:     Source inter-packet arrival time (ms)
      - dinpkt:     Destination inter-packet arrival time (ms)
      - sjit:       Source jitter (ms)
      - djit:       Destination jitter (ms)
      - tcprtt:     TCP connection round-trip time
      - ct_srv_src: Connections from same source to same service
      - ct_dst_ltm: Connections to same destination in last 100
      - label:      0 = Normal, 1 = Attack
      - attack_cat: Attack category name
    """
    np.random.seed(42)

    records = []

    # ── Generate NORMAL traffic ──
    for _ in range(n_normal):
        records.append({
            'dur':          np.random.exponential(2.0),          # Short flows
            'sbytes':       np.random.lognormal(7, 1.5),         # ~1KB typical
            'dbytes':       np.random.lognormal(8, 1.5),         # Server sends more
            'sttl':         np.random.choice([62, 63, 64, 128, 254, 255]),  # Standard TTLs
            'dttl':         np.random.choice([62, 63, 64, 128, 252, 254]),
            'sloss':        np.random.poisson(0.5),              # Very few drops
            'dloss':        np.random.poisson(0.3),
            'sload':        np.random.lognormal(10, 2),
            'dload':        np.random.lognormal(11, 2),
            'spkts':        np.random.poisson(8) + 1,            # ~8 packets
            'dpkts':        np.random.poisson(10) + 1,
            'sinpkt':       np.random.exponential(100),          # ~100ms spacing
            'dinpkt':       np.random.exponential(80),
            'sjit':         np.random.exponential(20),           # Low jitter
            'djit':         np.random.exponential(15),
            'tcprtt':       np.random.exponential(0.05),         # ~50ms RTT
            'ct_srv_src':   np.random.poisson(3) + 1,            # Few connections
            'ct_dst_ltm':   np.random.poisson(5) + 1,
            'proto':        np.random.choice(['tcp', 'udp', 'icmp'], p=[0.7, 0.25, 0.05]),
            'service':      np.random.choice(['http', 'https', 'dns', 'ssh', 'ftp', '-'],
                                              p=[0.3, 0.3, 0.15, 0.1, 0.05, 0.1]),
            'state':        np.random.choice(['FIN', 'CON', 'INT', 'RST'], p=[0.5, 0.3, 0.1, 0.1]),
            'label':        0,
            'attack_cat':   'Normal'
        })

    # ── Generate ATTACK traffic ──
    attack_configs = {
        'DoS': {
            'count': int(n_attack * 0.25),
            'dur': lambda: np.random.exponential(0.1),            # Very short bursts
            'sbytes': lambda: np.random.lognormal(5, 0.5),        # Small packets
            'spkts': lambda: np.random.poisson(500) + 100,        # Massive packet count!
            'sloss': lambda: np.random.poisson(20),               # Many drops
            'ct_srv_src': lambda: np.random.poisson(50) + 10,     # Many connections
        },
        'Reconnaissance': {
            'count': int(n_attack * 0.20),
            'dur': lambda: np.random.exponential(0.01),           # Quick probes
            'sbytes': lambda: np.random.lognormal(4, 0.5),        # Tiny packets
            'spkts': lambda: np.random.poisson(2) + 1,            # Few packets per probe
            'ct_dst_ltm': lambda: np.random.poisson(100) + 50,    # Scans MANY destinations
            'ct_srv_src': lambda: np.random.poisson(80) + 20,
        },
        'Exploit': {
            'count': int(n_attack * 0.20),
            'dur': lambda: np.random.exponential(5.0),            # Longer sessions
            'sbytes': lambda: np.random.lognormal(10, 2),         # Large payloads
            'dbytes': lambda: np.random.lognormal(12, 2),         # Large responses
            'sloss': lambda: np.random.poisson(5),
            'tcprtt': lambda: np.random.exponential(0.5),         # Higher latency
        },
        'Backdoor': {
            'count': int(n_attack * 0.10),
            'dur': lambda: np.random.exponential(60.0),           # Long-lived connections
            'sbytes': lambda: np.random.lognormal(6, 1),
            'sinpkt': lambda: np.random.exponential(5000),        # Very slow, stealthy
            'dinpkt': lambda: np.random.exponential(5000),
        },
        'Fuzzers': {
            'count': int(n_attack * 0.15),
            'dur': lambda: np.random.exponential(1.0),
            'sbytes': lambda: np.random.lognormal(9, 3),          # Random sizes
            'spkts': lambda: np.random.poisson(30) + 10,
            'sjit': lambda: np.random.exponential(200),           # High jitter
            'djit': lambda: np.random.exponential(150),
            'sloss': lambda: np.random.poisson(10),               # Many errors
        },
        'Generic': {
            'count': n_attack - int(n_attack * 0.90),  # Remaining
            'dur': lambda: np.random.exponential(3.0),
            'sbytes': lambda: np.random.lognormal(8, 2),
            'spkts': lambda: np.random.poisson(20) + 5,
            'ct_srv_src': lambda: np.random.poisson(20) + 5,
        }
    }

    for attack_name, cfg in attack_configs.items():
        for _ in range(cfg['count']):
            # Start with normal baseline, then override with attack characteristics
            record = {
                'dur':          cfg.get('dur', lambda: np.random.exponential(2.0))(),
                'sbytes':       cfg.get('sbytes', lambda: np.random.lognormal(7, 1.5))(),
                'dbytes':       cfg.get('dbytes', lambda: np.random.lognormal(8, 1.5))(),
                'sttl':         np.random.choice([62, 63, 64, 128, 254, 255]),
                'dttl':         np.random.choice([62, 63, 64, 128, 252, 254]),
                'sloss':        cfg.get('sloss', lambda: np.random.poisson(0.5))(),
                'dloss':        np.random.poisson(1),
                'sload':        np.random.lognormal(10, 2),
                'dload':        np.random.lognormal(11, 2),
                'spkts':        cfg.get('spkts', lambda: np.random.poisson(8) + 1)(),
                'dpkts':        np.random.poisson(10) + 1,
                'sinpkt':       cfg.get('sinpkt', lambda: np.random.exponential(100))(),
                'dinpkt':       cfg.get('dinpkt', lambda: np.random.exponential(80))(),
                'sjit':         cfg.get('sjit', lambda: np.random.exponential(20))(),
                'djit':         cfg.get('djit', lambda: np.random.exponential(15))(),
                'tcprtt':       cfg.get('tcprtt', lambda: np.random.exponential(0.05))(),
                'ct_srv_src':   cfg.get('ct_srv_src', lambda: np.random.poisson(3) + 1)(),
                'ct_dst_ltm':   cfg.get('ct_dst_ltm', lambda: np.random.poisson(5) + 1)(),
                'proto':        np.random.choice(['tcp', 'udp', 'icmp'], p=[0.8, 0.15, 0.05]),
                'service':      np.random.choice(['http', 'https', 'dns', 'ssh', 'ftp', '-'],
                                                  p=[0.25, 0.2, 0.1, 0.15, 0.1, 0.2]),
                'state':        np.random.choice(['FIN', 'CON', 'INT', 'RST', 'REQ'],
                                                  p=[0.2, 0.2, 0.2, 0.3, 0.1]),
                'label':        1,
                'attack_cat':   attack_name
            }
            records.append(record)

    df = pd.DataFrame(records)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

    print(f"   Generated: {len(df)} records ({n_normal} normal, {n_attack} attack)")
    print(f"   Attack distribution:")
    for cat, count in df[df['label']==1]['attack_cat'].value_counts().items():
        print(f"     - {cat}: {count}")

    return df

In [3]:
def load_unsw_nb15(csv_path: str = None) -> pd.DataFrame:
    """
    Load the UNSW-NB15 dataset.

    Download from:
      https://research.unsw.edu.au/projects/unsw-nb15-dataset
      OR
      https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15

    The dataset has these files:
      - UNSW-NB15_1.csv, UNSW-NB15_2.csv, UNSW-NB15_3.csv, UNSW-NB15_4.csv
      - UNSW-NB15_features.csv (column descriptions)
      - UNSW-NB15_GT.csv (ground truth)

    For simplicity, we use the pre-split training/testing CSVs:
      - UNSW_NB15_training-set.csv  (175,341 records)
      - UNSW_NB15_testing-set.csv   (82,332 records)

    If no path provided, we generate a realistic synthetic version
    for demonstration (same feature structure).
    """

    if csv_path and os.path.exists(csv_path):
        print(f"📂 Loading dataset from: {csv_path}")
        df = pd.read_csv(csv_path)
        print(f"   Shape: {df.shape}")
        return df

    print("📦 No dataset file found. Generating synthetic UNSW-NB15-style data...")
    print("   ⚠️  For your dissertation, download the REAL dataset from:")
    print("      https://research.unsw.edu.au/projects/unsw-nb15-dataset")
    print("      or Kaggle: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15")
    print()

    return generate_synthetic_dataset()

In [4]:
def preprocess(df: pd.DataFrame) -> tuple:
    """
    Clean and prepare the dataset for ML models.

    Steps:
      1. Handle missing values
      2. Encode categorical features (proto, service, state)
      3. Remove infinite values
      4. Separate features (X) from labels (y)
      5. Normalize all features to 0-1 range

    Returns:
      X:            Numpy array of features (normalized)
      y:            Numpy array of labels (0=normal, 1=attack)
      feature_names: List of feature column names
      scaler:       Fitted StandardScaler (save this for inference later!)
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    print("\n🔧 PREPROCESSING")
    print("=" * 50)

    df = df.copy()

    # ── 1. Handle missing values ──
    missing_before = df.isnull().sum().sum()

    # Fill numeric columns with median
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

    # Fill categorical columns with mode
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if col not in ['label', 'attack_cat']:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')

    missing_after = df.isnull().sum().sum()
    print(f"  Missing values: {missing_before} → {missing_after}")

    # ── 2. Encode categorical features ──
    # One-hot encode protocol, service, state
    encode_cols = [c for c in ['proto', 'service', 'state'] if c in df.columns]
    if encode_cols:
        df = pd.get_dummies(df, columns=encode_cols, drop_first=False)
        print(f"  One-hot encoded: {encode_cols}")

    # ── 3. Remove infinite values ──
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)

    # ── 4. Separate features and labels ──
    # Drop non-feature columns
    drop_cols = ['label', 'attack_cat', 'id']
    drop_cols = [c for c in drop_cols if c in df.columns]

    y = df['label'].values.astype(int)
    attack_cats = df['attack_cat'].values if 'attack_cat' in df.columns else None

    X_df = df.drop(columns=drop_cols)
    feature_names = X_df.columns.tolist()
    X = X_df.values.astype(np.float64)

    # ── 5. Normalize features ──
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    print(f"  Feature matrix shape: {X.shape}")
    print(f"  Label distribution: Normal={np.sum(y==0)}, Attack={np.sum(y==1)}")
    print(f"  Attack ratio: {np.mean(y)*100:.1f}%")

    return X, y, feature_names, scaler, attack_cats

In [5]:
def split_data(X: np.ndarray, y: np.ndarray,
               train_ratio: float = 0.7,
               random_state: int = 42) -> dict:
    """
    IMPORTANT: For anomaly detection, the training set should be
    MOSTLY or ENTIRELY normal data. The test set contains both
    normal and attack data.

    Strategy:
      1. Separate normal and attack samples
      2. Use 70% of NORMAL data for training
      3. Remaining 30% normal + ALL attacks = test set

    Why? Because Isolation Forest and One-Class SVM learn what
    "normal" looks like. They've never seen attacks during training.
    At test time, attacks appear as outliers.

    Returns dict with:
      X_train, y_train  — Training set (normal data only)
      X_test, y_test    — Test set (mixed normal + attacks)
    """
    np.random.seed(random_state)

    print("\n✂️  DATA SPLITTING")
    print("=" * 50)

    # Separate normal and attack indices
    normal_idx = np.where(y == 0)[0]
    attack_idx = np.where(y == 1)[0]

    # Shuffle normal indices
    np.random.shuffle(normal_idx)

    # Split normal data: 70% train, 30% test
    split_point = int(len(normal_idx) * train_ratio)
    train_normal_idx = normal_idx[:split_point]
    test_normal_idx = normal_idx[split_point:]

    # Training set: ONLY normal data
    X_train = X[train_normal_idx]
    y_train = y[train_normal_idx]   # All zeros

    # Test set: Remaining normal + ALL attacks
    test_idx = np.concatenate([test_normal_idx, attack_idx])
    np.random.shuffle(test_idx)
    X_test = X[test_idx]
    y_test = y[test_idx]

    print(f"  Training set:  {len(X_train)} samples (100% normal)")
    print(f"  Test set:      {len(X_test)} samples "
          f"({np.sum(y_test==0)} normal + {np.sum(y_test==1)} attack)")
    print(f"  Test attack ratio: {np.mean(y_test)*100:.1f}%")

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': y_test,
        'train_idx': train_normal_idx, 'test_idx': test_idx
    }

In [6]:
def train_isolation_forest(X_train: np.ndarray, contamination: float = 0.05) -> object:
    """
    Isolation Forest works by randomly selecting a feature and a split value.
    Anomalies are isolated in fewer splits because they are "different" from
    the majority.

    Think of it like a game of 20 Questions:
      - Normal data points: Need many questions to isolate
      - Anomalies: Can be identified in just a few questions

    Parameters:
      n_estimators:  Number of isolation trees (more = better but slower)
      contamination: Expected % of anomalies (0.05 = 5%)
      max_samples:   Samples per tree (256 is usually enough)
    """
    from sklearn.ensemble import IsolationForest

    print("\n🌲 Training Isolation Forest...")

    model = IsolationForest(
        n_estimators=200,          # 200 trees
        contamination=contamination,
        max_samples=min(256, len(X_train)),  # Subsample for speed
        random_state=42,
        n_jobs=-1                  # Use all CPU cores
    )

    model.fit(X_train)
    print(f"   ✅ Trained on {len(X_train)} samples with {model.n_estimators} trees")

    return model


In [7]:
def train_one_class_svm(X_train: np.ndarray, nu: float = 0.05) -> object:
    """
    One-Class SVM learns a "boundary" around normal data in high-dimensional space.
    Anything outside the boundary = anomaly.

    Uses RBF (Radial Basis Function) kernel to handle non-linear boundaries.

    Parameters:
      kernel: 'rbf' for non-linear boundaries (recommended)
      gamma:  Controls boundary tightness ('scale' = auto-calculated)
      nu:     Upper bound on fraction of outliers (0.05 = 5%)

    Note: SVM is SLOW on large datasets. We subsample if needed.
    """
    from sklearn.svm import OneClassSVM

    print("\n🔵 Training One-Class SVM...")

    # Subsample for SVM (it's O(n²) — very slow on large data)
    max_samples = 5000
    if len(X_train) > max_samples:
        print(f"   ⚠️  Subsampling {max_samples}/{len(X_train)} for SVM (O(n²) complexity)")
        idx = np.random.choice(len(X_train), max_samples, replace=False)
        X_svm = X_train[idx]
    else:
        X_svm = X_train

    model = OneClassSVM(
        kernel='rbf',
        gamma='scale',
        nu=nu
    )

    model.fit(X_svm)
    print(f"   ✅ Trained on {len(X_svm)} samples")

    return model

In [8]:
def train_autoencoder(X_train: np.ndarray,
                      encoding_dim: int = 8,
                      epochs: int = 50,
                      batch_size: int = 64) -> tuple:
    """
    Autoencoder learns to COMPRESS and RECONSTRUCT normal data.

    Architecture:
      Input (N features)  →  Encoder  →  Bottleneck (8 dims)  →  Decoder  →  Output (N features)

    Training: Minimize reconstruction error on NORMAL data.
    Detection: If reconstruction error is HIGH → the data doesn't look normal → anomaly!

    Think of it like:
      - Train someone to draw cats by showing them 1000 cat photos
      - Show them a dog → they try to draw it as a cat → it looks wrong
      - The "wrongness" (reconstruction error) tells you it's not a cat

    Returns:
      autoencoder:  The full autoencoder model
      threshold:    Reconstruction error threshold for anomaly detection
    """
    import tensorflow as tf
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping

    print("\n🧠 Training Autoencoder...")

    input_dim = X_train.shape[1]

    # Suppress TF logging
    tf.get_logger().setLevel('ERROR')
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

    # ── Build the Autoencoder ──
    # Encoder: input → 64 → 32 → 16 → 8 (bottleneck)
    # Decoder: 8 → 16 → 32 → 64 → input (reconstruction)

    input_layer = Input(shape=(input_dim,))

    # Encoder
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = BatchNormalization()(encoded)
    encoded = Dropout(0.2)(encoded)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = BatchNormalization()(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    encoded = Dense(encoding_dim, activation='relu', name='bottleneck')(encoded)

    # Decoder (mirror of encoder)
    decoded = Dense(16, activation='relu')(encoded)
    decoded = BatchNormalization()(decoded)
    decoded = Dense(32, activation='relu')(decoded)
    decoded = BatchNormalization()(decoded)
    decoded = Dropout(0.2)(decoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)   # Linear output for reconstruction

    autoencoder = Model(inputs=input_layer, outputs=decoded)

    autoencoder.compile(
        optimizer='adam',
        loss='mse'          # Mean Squared Error = reconstruction error
    )

    print(f"   Architecture: {input_dim} → 64 → 32 → 16 → {encoding_dim} → 16 → 32 → 64 → {input_dim}")
    print(f"   Total parameters: {autoencoder.count_params():,}")

    # ── Train ──
    # Use 10% of training data for validation (to detect overfitting)
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,          # Stop if no improvement for 5 epochs
        restore_best_weights=True
    )

    history = autoencoder.fit(
        X_train, X_train,    # Input = Output (reconstruct itself!)
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0             # Suppress epoch logs
    )

    best_epoch = len(history.history['loss']) - early_stop.patience
    print(f"   ✅ Trained for {len(history.history['loss'])} epochs "
          f"(best at epoch {max(1, best_epoch)})")

    # ── Calculate anomaly threshold ──
    # The threshold = 95th percentile of reconstruction error on NORMAL data
    # Anything above this = anomaly

    reconstructions = autoencoder.predict(X_train, verbose=0)
    reconstruction_errors = np.mean(np.square(X_train - reconstructions), axis=1)

    threshold = np.percentile(reconstruction_errors, 95)

    print(f"   Reconstruction error stats on normal data:")
    print(f"     Mean:  {np.mean(reconstruction_errors):.6f}")
    print(f"     Std:   {np.std(reconstruction_errors):.6f}")
    print(f"     95th:  {threshold:.6f}  ← ANOMALY THRESHOLD")

    return autoencoder, threshold, history

In [9]:
def evaluate_all_models(models: dict, X_test: np.ndarray, y_test: np.ndarray,
                        feature_names: list = None) -> dict:
    """
    Evaluate all three models and produce:
      - Classification report (precision, recall, F1)
      - Confusion matrix
      - ROC-AUC score
      - Comparison chart

    Returns a dict of results for each model.
    """
    from sklearn.metrics import (classification_report, confusion_matrix,
                                  roc_auc_score, roc_curve, f1_score,
                                  precision_score, recall_score)

    print("\n📊 MODEL EVALUATION")
    print("=" * 70)

    results = {}

    # ── Evaluate each model ──
    for name, model_info in models.items():
        print(f"\n── {name.upper()} ──")

        if name == 'autoencoder':
            # Autoencoder: Use reconstruction error
            autoencoder, threshold = model_info['model'], model_info['threshold']
            reconstructions = autoencoder.predict(X_test, verbose=0)
            errors = np.mean(np.square(X_test - reconstructions), axis=1)
            y_pred = (errors > threshold).astype(int)
            y_scores = errors  # Higher error = more anomalous

        else:
            # Isolation Forest / One-Class SVM: -1 = anomaly, 1 = normal
            model = model_info['model']
            raw_pred = model.predict(X_test)
            y_pred = (raw_pred == -1).astype(int)  # Convert: -1→1 (attack), 1→0 (normal)

            if hasattr(model, 'decision_function'):
                y_scores = -model.decision_function(X_test)  # Negate so higher = more anomalous
            else:
                y_scores = y_pred.astype(float)

        # ── Metrics ──
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        try:
            auc = roc_auc_score(y_test, y_scores)
        except:
            auc = 0.0

        cm = confusion_matrix(y_test, y_pred)

        print(classification_report(y_test, y_pred,
                                     target_names=['Normal', 'Attack'],
                                     zero_division=0))
        print(f"  ROC-AUC: {auc:.4f}")
        print(f"  Confusion Matrix:")
        print(f"    TN={cm[0][0]:5d}  FP={cm[0][1]:5d}")
        print(f"    FN={cm[1][0]:5d}  TP={cm[1][1]:5d}")

        results[name] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc,
            'confusion_matrix': cm,
            'y_pred': y_pred,
            'y_scores': y_scores
        }

    # ── Ensemble (majority vote) ──
    print(f"\n── ENSEMBLE (MAJORITY VOTE) ──")
    all_preds = np.stack([results[m]['y_pred'] for m in results])
    ensemble_pred = (np.sum(all_preds, axis=0) >= 2).astype(int)  # At least 2 out of 3 agree

    precision = precision_score(y_test, ensemble_pred, zero_division=0)
    recall = recall_score(y_test, ensemble_pred, zero_division=0)
    f1 = f1_score(y_test, ensemble_pred, zero_division=0)
    cm = confusion_matrix(y_test, ensemble_pred)

    print(classification_report(y_test, ensemble_pred,
                                 target_names=['Normal', 'Attack'],
                                 zero_division=0))
    print(f"  Confusion Matrix:")
    print(f"    TN={cm[0][0]:5d}  FP={cm[0][1]:5d}")
    print(f"    FN={cm[1][0]:5d}  TP={cm[1][1]:5d}")

    results['ensemble'] = {
        'precision': precision, 'recall': recall, 'f1': f1,
        'confusion_matrix': cm, 'y_pred': ensemble_pred
    }

    return results

In [10]:
def plot_results(results: dict, y_test: np.ndarray, save_path: str = './plots'):
    """Generate evaluation plots for the dissertation."""
    from sklearn.metrics import roc_curve

    os.makedirs(save_path, exist_ok=True)

    # ── Plot 1: Model Comparison Bar Chart ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    model_names = ['isolation_forest', 'one_class_svm', 'autoencoder', 'ensemble']
    display_names = ['Isolation Forest', 'One-Class SVM', 'Autoencoder', 'Ensemble']
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

    for idx, metric in enumerate(['precision', 'recall', 'f1']):
        values = [results[m][metric] for m in model_names]
        bars = axes[idx].bar(display_names, values, color=colors)
        axes[idx].set_title(f'{metric.upper()}', fontsize=14, fontweight='bold')
        axes[idx].set_ylim(0, 1.05)
        axes[idx].grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, values):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                          f'{val:.3f}', ha='center', fontsize=10)

    plt.suptitle('CloudTwin AI — Model Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{save_path}/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📈 Saved: {save_path}/model_comparison.png")

    # ── Plot 2: ROC Curves ──
    fig, ax = plt.subplots(figsize=(8, 6))

    for name, display, color in zip(
        ['isolation_forest', 'one_class_svm', 'autoencoder'],
        ['Isolation Forest', 'One-Class SVM', 'Autoencoder'],
        ['#2196F3', '#FF9800', '#4CAF50']
    ):
        if 'y_scores' in results[name]:
            fpr, tpr, _ = roc_curve(y_test, results[name]['y_scores'])
            auc = results[name].get('auc', 0)
            ax.plot(fpr, tpr, color=color, lw=2, label=f'{display} (AUC={auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('CloudTwin AI — ROC Curves', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{save_path}/roc_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📈 Saved: {save_path}/roc_curves.png")

    # ── Plot 3: Confusion Matrices ──
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    for idx, (name, display) in enumerate(zip(model_names, display_names)):
        cm = results[name]['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['Normal', 'Attack'],
                    yticklabels=['Normal', 'Attack'])
        axes[idx].set_title(display, fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Actual')
        axes[idx].set_xlabel('Predicted')

    plt.suptitle('CloudTwin AI — Confusion Matrices', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{save_path}/confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📈 Saved: {save_path}/confusion_matrices.png")


In [11]:
def plot_autoencoder_training(history, save_path: str = './plots'):
    """Plot autoencoder training loss curve."""
    os.makedirs(save_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.history['loss'], label='Training Loss', color='#2196F3', lw=2)
    ax.plot(history.history['val_loss'], label='Validation Loss', color='#FF9800', lw=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Mean Squared Error', fontsize=12)
    ax.set_title('Autoencoder Training Loss', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{save_path}/autoencoder_loss.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📈 Saved: {save_path}/autoencoder_loss.png")



In [12]:
def save_models(models: dict, scaler, feature_names: list,
                save_dir: str = './saved_models'):
    """
    Save trained models so they can be loaded later for inference
    (when the actual CloudTwin system is running).
    """
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n💾 SAVING MODELS to {save_dir}/")
    print("=" * 50)

    # Save Isolation Forest
    joblib.dump(models['isolation_forest']['model'],
                f'{save_dir}/isolation_forest.joblib')
    print(f"  ✅ isolation_forest.joblib")

    # Save One-Class SVM
    joblib.dump(models['one_class_svm']['model'],
                f'{save_dir}/one_class_svm.joblib')
    print(f"  ✅ one_class_svm.joblib")

    # Save Autoencoder
    models['autoencoder']['model'].save(f'{save_dir}/autoencoder.keras')
    print(f"  ✅ autoencoder.keras")

    # Save threshold
    with open(f'{save_dir}/autoencoder_threshold.json', 'w') as f:
        json.dump({'threshold': float(models['autoencoder']['threshold'])}, f)
    print(f"  ✅ autoencoder_threshold.json")

    # Save scaler (CRITICAL — need this for preprocessing new data!)
    joblib.dump(scaler, f'{save_dir}/scaler.joblib')
    print(f"  ✅ scaler.joblib")

    # Save feature names
    with open(f'{save_dir}/feature_names.json', 'w') as f:
        json.dump(feature_names, f)
    print(f"  ✅ feature_names.json")

    print(f"\n  📦 All models saved! To load later:")
    print(f"     if_model = joblib.load('{save_dir}/isolation_forest.joblib')")
    print(f"     svm_model = joblib.load('{save_dir}/one_class_svm.joblib')")
    print(f"     ae_model = tf.keras.models.load_model('{save_dir}/autoencoder.keras')")


In [13]:
def export_detections_for_audit_ledger(results: dict, y_test: np.ndarray,
                                        save_path: str = './detections.json') -> list:
    """
    Export detected anomalies in a format that Component 4 (Audit Ledger)
    can ingest directly.
    """
    ensemble_pred = results['ensemble']['y_pred']

    detections = []
    for i in range(len(y_test)):
        if ensemble_pred[i] == 1:  # Detected as anomaly
            detection = {
                "alert_id": f"ML-ALERT-{len(detections)+1:04d}",
                "timestamp": datetime.now().isoformat(),
                "test_sample_index": int(i),
                "actual_label": "attack" if y_test[i] == 1 else "normal",
                "predicted_label": "attack",
                "risk_level": "HIGH" if y_test[i] == 1 else "MEDIUM",
                "detection_agreement": {
                    "isolation_forest": bool(results['isolation_forest']['y_pred'][i]),
                    "one_class_svm": bool(results['one_class_svm']['y_pred'][i]),
                    "autoencoder": bool(results['autoencoder']['y_pred'][i])
                }
            }
            detections.append(detection)

    with open(save_path, 'w') as f:
        json.dump(detections, f, indent=2)

    true_pos = sum(1 for d in detections if d['actual_label'] == 'attack')
    false_pos = sum(1 for d in detections if d['actual_label'] == 'normal')

    print(f"\n📦 EXPORTED FOR AUDIT LEDGER")
    print(f"  Total detections: {len(detections)}")
    print(f"  True positives:   {true_pos}")
    print(f"  False positives:  {false_pos}")
    print(f"  Saved to: {save_path}")

    return detections


In [14]:
def main():
    print("╔" + "═" * 68 + "╗")
    print("║" + "  CLOUDTWIN AI — ML TRAINING PIPELINE  ".center(68) + "║")
    print("╚" + "═" * 68 + "╝")
    print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # ────────────────────────────────────────────────────────────────────
    # STEP 1: Load data
    # ────────────────────────────────────────────────────────────────────
    # TO USE REAL DATA: Change this path to your downloaded CSV
    df = load_unsw_nb15("/content/UNSW_NB15_training-set.csv")
    #df = load_unsw_nb15()  # Uses synthetic data for demo

    # ────────────────────────────────────────────────────────────────────
    # STEP 2: Preprocess
    # ────────────────────────────────────────────────────────────────────
    X, y, feature_names, scaler, attack_cats = preprocess(df)

    # ────────────────────────────────────────────────────────────────────
    # STEP 3: Split
    # ────────────────────────────────────────────────────────────────────
    splits = split_data(X, y)
    X_train = splits['X_train']
    X_test = splits['X_test']
    y_test = splits['y_test']

    # ────────────────────────────────────────────────────────────────────
    # STEP 4: Train all 3 models
    # ────────────────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  TRAINING MODELS")
    print("=" * 70)

    # Model 1: Isolation Forest
    if_model = train_isolation_forest(X_train, contamination=0.05)

    # Model 2: One-Class SVM
    svm_model = train_one_class_svm(X_train, nu=0.05)

    # Model 3: Autoencoder
    ae_model, ae_threshold, ae_history = train_autoencoder(
        X_train, encoding_dim=8, epochs=50, batch_size=64
    )

    # Package models
    models = {
        'isolation_forest': {'model': if_model},
        'one_class_svm': {'model': svm_model},
        'autoencoder': {'model': ae_model, 'threshold': ae_threshold}
    }

    # ────────────────────────────────────────────────────────────────────
    # STEP 5: Evaluate
    # ────────────────────────────────────────────────────────────────────
    results = evaluate_all_models(models, X_test, y_test, feature_names)

    # Generate plots
    print("\n🎨 GENERATING PLOTS")
    print("=" * 50)
    plot_results(results, y_test)
    plot_autoencoder_training(ae_history)

    # ────────────────────────────────────────────────────────────────────
    # STEP 6: Save & Export
    # ────────────────────────────────────────────────────────────────────
    save_models(models, scaler, feature_names)
    detections = export_detections_for_audit_ledger(results, y_test)

    # ────────────────────────────────────────────────────────────────────
    # Final Summary
    # ────────────────────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("  TRAINING PIPELINE COMPLETE")
    print("═" * 70)
    print(f"  Dataset:          {len(df)} records")
    print(f"  Training samples: {len(X_train)} (normal only)")
    print(f"  Test samples:     {len(X_test)} (mixed)")
    print()
    print(f"  ┌─────────────────┬───────────┬────────┬────────┬─────────┐")
    print(f"  │ Model           │ Precision │ Recall │   F1   │ ROC-AUC │")
    print(f"  ├─────────────────┼───────────┼────────┼────────┼─────────┤")
    for name, display in [('isolation_forest', 'Isolation Forest'),
                          ('one_class_svm', 'One-Class SVM'),
                          ('autoencoder', 'Autoencoder'),
                          ('ensemble', 'Ensemble')]:
        r = results[name]
        auc = r.get('auc', '-')
        auc_str = f"{auc:.3f}" if isinstance(auc, float) else "  -  "
        print(f"  │ {display:15s} │   {r['precision']:.3f}   │ {r['recall']:.3f}  │ {r['f1']:.3f}  │ {auc_str} │")
    print(f"  └─────────────────┴───────────┴────────┴────────┴─────────┘")
    print()
    print(f"  Detections exported: {len(detections)} → detections.json")
    print(f"  Models saved to:     ./saved_models/")
    print(f"  Plots saved to:      ./plots/")
    print("═" * 70)

In [15]:
if __name__ == "__main__":
    main()


╔════════════════════════════════════════════════════════════════════╗
║                CLOUDTWIN AI — ML TRAINING PIPELINE                 ║
╚════════════════════════════════════════════════════════════════════╝
  Started: 2026-03-12 05:20:58
📂 Loading dataset from: /content/UNSW_NB15_training-set.csv
   Shape: (82332, 45)

🔧 PREPROCESSING
  Missing values: 0 → 0
  One-hot encoded: ['proto', 'service', 'state']
  Feature matrix shape: (82332, 190)
  Label distribution: Normal=37000, Attack=45332
  Attack ratio: 55.1%

✂️  DATA SPLITTING
  Training set:  25900 samples (100% normal)
  Test set:      56432 samples (11100 normal + 45332 attack)
  Test attack ratio: 80.3%

  TRAINING MODELS

🌲 Training Isolation Forest...
   ✅ Trained on 25900 samples with 200 trees

🔵 Training One-Class SVM...
   ⚠️  Subsampling 5000/25900 for SVM (O(n²) complexity)
   ✅ Trained on 5000 samples

🧠 Training Autoencoder...
   Architecture: 190 → 64 → 32 → 16 → 8 → 16 → 32 → 64 → 190
   Total parameters: 30,